In [1]:
!pip install segmentation-models-pytorch
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import InterpolationMode
from torchvision import transforms
from torchvision import models
import segmentation_models_pytorch as smp
from tqdm import tqdm
from torchmetrics.classification import BinaryJaccardIndex

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
# ============================================
# 1. PATHS
# ============================================
images_train_path = '/kaggle/input/brisc2025/brisc2025/segmentation_task/train/images'
masks_train_path = '/kaggle/input/brisc2025/brisc2025/segmentation_task/train/masks'

train_image_paths = sorted([os.path.join(images_train_path, f) for f in os.listdir(images_train_path) if 't1' in f])
train_target_paths = sorted([os.path.join(masks_train_path, f) for f in os.listdir(masks_train_path) if 't1' in f])

# ============================================
# 2. PROPER TRAIN/VAL SPLIT
# ============================================
train_tumors_type = [os.path.basename(path).split('_')[3] for path in train_image_paths]
train_image_paths, val_image_paths, train_target_paths, val_target_paths = train_test_split(
    train_image_paths, train_target_paths, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_tumors_type
)

# Test set (keep separate!)
images_test_path = '/kaggle/input/brisc2025/brisc2025/segmentation_task/test/images'
masks_test_path = '/kaggle/input/brisc2025/brisc2025/segmentation_task/test/masks'
test_image_paths = sorted([os.path.join(images_test_path, f) for f in os.listdir(images_test_path) if 't1' in f])
test_target_paths = sorted([os.path.join(masks_test_path, f) for f in os.listdir(masks_test_path) if 't1' in f])

# Verification
print(f"Train: {len(train_image_paths)} images")
print(f"Val: {len(val_image_paths)} images")
print(f"Test: {len(test_image_paths)} images")

Train: 3146 images
Val: 787 images
Test: 860 images


In [4]:
# ============================================
# 3. CALCULATE MEAN/STD
# ============================================
print("\nCalculating dataset statistics...")
total_sum = np.zeros(3)
total_pixels = 0

for image_path in tqdm(train_image_paths, desc="Computing mean"):
    image = np.array(Image.open(image_path).convert("RGB")) / 255.0
    total_sum += np.sum(image, axis=(0, 1))
    total_pixels += image.shape[0] * image.shape[1]

mean = total_sum / total_pixels

total_squared_diff = np.zeros(3)
for image_path in tqdm(train_image_paths, desc="Computing std"):
    image = np.array(Image.open(image_path).convert("RGB")) / 255.0
    total_squared_diff += np.sum((image - mean) ** 2, axis=(0, 1))

std = np.sqrt(total_squared_diff / total_pixels)

print(f"Dataset Mean: {mean}")
print(f"Dataset Std: {std}")


Calculating dataset statistics...


Computing std: 100%|██████████| 3146/3146 [00:29<00:00, 107.42it/s]

Dataset Mean: [0.16475258 0.16475258 0.16475258]
Dataset Std: [0.17378858 0.17378858 0.17378858]


In [5]:
# ============================================
# 4. DATASET CLASS
# ============================================
class BRISCCustomDataset(Dataset):
    def __init__(self, image_paths, target_paths, img_transform=None, mask_transform=None):
        self.img_transform = img_transform
        self.mask_transform = mask_transform
        self.image_paths = image_paths
        self.target_paths = target_paths

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")
        mask = Image.open(self.target_paths[index]).convert("L")
        
        image = self.img_transform(image)
        mask = self.mask_transform(mask)
       

        filename = os.path.basename(self.image_paths[index])
        tumor_type = filename.split('_')[3]
        
        return image, mask, tumor_type

    def __len__(self):
        return len(self.image_paths)

# ============================================
# 5. TRANSFORMS
# ============================================
img_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

mask_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=InterpolationMode.NEAREST), 
    transforms.ToTensor(), 
    transforms.Lambda(lambda x: (x > 0.5).float())
])

# ============================================
# 6. DATALOADERS
# ============================================
train_dataset = BRISCCustomDataset(train_image_paths, train_target_paths, img_transform, mask_transform)
val_dataset = BRISCCustomDataset(val_image_paths, val_target_paths, img_transform, mask_transform)
test_dataset = BRISCCustomDataset(test_image_paths, test_target_paths, img_transform, mask_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)


In [6]:
# ============================================
# 7. CNN MODEL DEFINITION
# ============================================
class CNNSegmentationModel(nn.Module):
    """
    CNN-based segmentation model using ResNet34 backbone
    """
    def __init__(self, num_classes=1):
        super(CNNSegmentationModel, self).__init__()
        
        # Encoder: ResNet34 pretrained
        resnet = models.resnet34(pretrained=True)
        self.encoder = nn.Sequential(*list(resnet.children())[:-2])
        
        # Decoder: Progressive upsampling
        self.decoder = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
        )
        
        self.segmentation_head = nn.Conv2d(16, num_classes, kernel_size=1)
        
    def forward(self, x):
        features = self.encoder(x)
        x = self.decoder(features)
        output = self.segmentation_head(x)
        return output

In [7]:
# ============================================
# 8. TRAINING FUNCTION
# ============================================
def train_model(model, model_name, train_loader, val_loader, device, epochs=20, save_path="best_model.pth"):
    """
    Generic training function for any segmentation model
    """
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    iou_metric = BinaryJaccardIndex().to(device)
    
    best_val_miou = 0.0
    
    print("\n" + "="*50)
    print(f"Starting {model_name} Training...")
    print("="*50)
    
    for epoch in range(epochs):
        # TRAINING
        model.train()
        total_train_loss = 0
        gl_train_iou, me_train_iou, pi_train_iou = [], [], []

        for images, masks, tumors_type in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Train"):
            images, masks = images.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            
            for i in range(images.size(0)):
                train_iou = iou_metric(preds[i].unsqueeze(0).int(), masks[i].unsqueeze(0).int()).item()
                if tumors_type[i] == 'gl':
                    gl_train_iou.append(train_iou)
                elif tumors_type[i] == 'me':
                    me_train_iou.append(train_iou)
                elif tumors_type[i] == 'pi':
                    pi_train_iou.append(train_iou)

        # Calculate weighted mIoU
        len_gl = len(gl_train_iou)
        len_me = len(me_train_iou)
        len_pi = len(pi_train_iou)
        total_samples = len_gl + len_me + len_pi
        
        gl_miou = np.mean(gl_train_iou) if len_gl > 0 else 0
        me_miou = np.mean(me_train_iou) if len_me > 0 else 0
        pi_miou = np.mean(pi_train_iou) if len_pi > 0 else 0
        weighted_miou = (gl_miou * len_gl + me_miou * len_me + pi_miou * len_pi) / total_samples if total_samples > 0 else 0
        
        print(f'\n[TRAIN] Loss: {total_train_loss / len(train_loader):.4f}')
        print(f'GL: {gl_miou:.4f} ({len_gl}) | ME: {me_miou:.4f} ({len_me}) | PI: {pi_miou:.4f} ({len_pi}) | Weighted: {weighted_miou:.4f}')

        # VALIDATION
        model.eval()
        total_val_loss = 0
        gl_val_iou, me_val_iou, pi_val_iou = [], [], []
        
        with torch.no_grad():
            for images, masks, tumors_type in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Val"):
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                total_val_loss += loss.item()
                
                preds = (torch.sigmoid(outputs) > 0.5).float()
                for i in range(images.size(0)):
                    val_iou = iou_metric(preds[i].unsqueeze(0).int(), masks[i].unsqueeze(0).int()).item()
                    if tumors_type[i] == 'gl':
                        gl_val_iou.append(val_iou)
                    elif tumors_type[i] == 'me':
                        me_val_iou.append(val_iou)
                    elif tumors_type[i] == 'pi':
                        pi_val_iou.append(val_iou)

        len_gl = len(gl_val_iou)
        len_me = len(me_val_iou)
        len_pi = len(pi_val_iou)
        total_samples = len_gl + len_me + len_pi
        
        gl_miou = np.mean(gl_val_iou) if len_gl > 0 else 0
        me_miou = np.mean(me_val_iou) if len_me > 0 else 0
        pi_miou = np.mean(pi_val_iou) if len_pi > 0 else 0
        weighted_miou = (gl_miou * len_gl + me_miou * len_me + pi_miou * len_pi) / total_samples if total_samples > 0 else 0
        
        print(f'[VAL] Loss: {total_val_loss / len(val_loader):.4f}')
        print(f'GL: {gl_miou:.4f} ({len_gl}) | ME: {me_miou:.4f} ({len_me}) | PI: {pi_miou:.4f} ({len_pi}) | Weighted: {weighted_miou:.4f}')
        
        # Save best model
        if weighted_miou > best_val_miou:
            best_val_miou = weighted_miou
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_miou': weighted_miou,
            }, save_path)
            print(f"✓ Saved new best {model_name} model! (Weighted mIoU: {weighted_miou:.4f})")
        
        scheduler.step(total_val_loss / len(val_loader))
        print(f"Current LR: {optimizer.param_groups[0]['lr']:.2e}")
        print("="*50)

    print(f"\n{model_name} Training complete! Best validation mIoU: {best_val_miou:.4f}")
    return best_val_miou

In [8]:
# ============================================
# 9. TRAIN U-NET
# ============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    unet_model = torch.nn.DataParallel(unet_model)

unet_best_miou = train_model(
    model=unet_model,
    model_name="U-Net",
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=20,
    save_path="best_unet_model.pth"
)


Using device: cuda
Number of GPUs: 2


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Using 2 GPUs with DataParallel

Starting U-Net Training...


Epoch 1/20 - Train: 100%|██████████| 197/197 [00:37<00:00,  5.27it/s]



[TRAIN] Loss: 0.2315
GL: 0.1612 (918) | ME: 0.3573 (1063) | PI: 0.2691 (1165) | Weighted: 0.2674


... (74 outputs left out)

Epoch 20/20 - Train: 100%|██████████| 197/197 [00:36<00:00,  5.47it/s]



[TRAIN] Loss: 0.0045
GL: 0.8664 (918) | ME: 0.9187 (1063) | PI: 0.8758 (1165) | Weighted: 0.8875


Epoch 20/20 - Val: 100%|██████████| 50/50 [00:04<00:00, 10.59it/s]

[VAL] Loss: 0.0142
GL: 0.6235 (229) | ME: 0.8767 (266) | PI: 0.7748 (292) | Weighted: 0.7652
Current LR: 1.00e-04

U-Net Training complete! Best validation mIoU: 0.7699


In [9]:
# Free up memory
del unet_model
torch.cuda.empty_cache()

In [10]:
# ============================================
# 10. TRAIN CNN
# ============================================
cnn_model = CNNSegmentationModel(num_classes=1).to(device)

if torch.cuda.device_count() > 1:
    cnn_model = torch.nn.DataParallel(cnn_model)

cnn_best_miou = train_model(
    model=cnn_model,
    model_name="CNN",
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=20,
    save_path="best_cnn_model.pth"
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 172MB/s] 



Starting CNN Training...


Epoch 1/20 - Train: 100%|██████████| 197/197 [00:26<00:00,  7.33it/s]



[TRAIN] Loss: 0.4676
GL: 0.0952 (918) | ME: 0.2054 (1063) | PI: 0.1393 (1165) | Weighted: 0.1488


... (74 outputs left out)

Epoch 20/20 - Train: 100%|██████████| 197/197 [00:26<00:00,  7.39it/s]



[TRAIN] Loss: 0.0674
GL: 0.7452 (918) | ME: 0.8190 (1063) | PI: 0.7473 (1165) | Weighted: 0.7709


Epoch 20/20 - Val: 100%|██████████| 50/50 [00:04<00:00, 11.55it/s]


[VAL] Loss: 0.0682
GL: 0.5518 (229) | ME: 0.7837 (266) | PI: 0.6804 (292) | Weighted: 0.6779
✓ Saved new best CNN model! (Weighted mIoU: 0.6779)
Current LR: 1.00e-04

CNN Training complete! Best validation mIoU: 0.6779


In [11]:
# Free up memory
del cnn_model
torch.cuda.empty_cache()

In [16]:
# ============================================
# 11. ENSEMBLE CLASS
# ============================================
class EnsembleModel:
    def __init__(self, unet_path, cnn_path, device, fusion_method='average'):
        """
        fusion_method: 'average', 'weighted', 'max'
        """
        self.device = device
        self.fusion_method = fusion_method
        
        # Load U-Net
        print("Loading U-Net...")
        self.unet = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=None,
            in_channels=3,
            classes=1,
            activation=None
        ).to(device)
        
        checkpoint = torch.load(unet_path, map_location=device, weights_only=False)  # ← FIXED
        self.unet.load_state_dict(checkpoint['model_state_dict'])
        self.unet.eval()
        print(f"✓ U-Net loaded (Val mIoU: {checkpoint['val_miou']:.4f})")
        
        # Load CNN
        print("Loading CNN...")
        self.cnn = CNNSegmentationModel(num_classes=1).to(device)
        
        checkpoint = torch.load(cnn_path, map_location=device, weights_only=False)  # ← FIXED
        self.cnn.load_state_dict(checkpoint['model_state_dict'])
        self.cnn.eval()
        print(f"✓ CNN loaded (Val mIoU: {checkpoint['val_miou']:.4f})")
        
    def predict(self, x, return_individual=False):
        with torch.no_grad():
            unet_logits = self.unet(x)
            cnn_logits = self.cnn(x)
            
            unet_probs = torch.sigmoid(unet_logits)
            cnn_probs = torch.sigmoid(cnn_logits)
            
            if self.fusion_method == 'average':
                ensemble_probs = (unet_probs + cnn_probs) / 2.0
            elif self.fusion_method == 'weighted':
                ensemble_probs = 0.6 * unet_probs + 0.4 * cnn_probs
            elif self.fusion_method == 'max':
                ensemble_probs = torch.max(unet_probs, cnn_probs)
            else:
                raise ValueError(f"Unknown fusion method: {self.fusion_method}")
            
            ensemble_preds = (ensemble_probs > 0.5).float()
            
            if return_individual:
                unet_preds = (unet_probs > 0.5).float()
                cnn_preds = (cnn_probs > 0.5).float()
                return ensemble_preds, unet_preds, cnn_preds
            
            return ensemble_preds


In [17]:
# ============================================
# 12. EVALUATION FUNCTION
# ============================================
def evaluate_model(model_or_ensemble, data_loader, device, model_name="Model"):
    iou_metric = BinaryJaccardIndex().to(device)
    gl_iou, me_iou, pi_iou = [], [], []
    
    # Check if it's an ensemble or regular model
    is_ensemble = isinstance(model_or_ensemble, EnsembleModel)
    
    if not is_ensemble:
        model_or_ensemble.eval()
    
    with torch.no_grad():
        for images, masks, tumors_type in tqdm(data_loader, desc=f"Evaluating {model_name}"):
            images, masks = images.to(device), masks.to(device)
            
            if is_ensemble:
                preds = model_or_ensemble.predict(images)
            else:
                outputs = model_or_ensemble(images)
                preds = (torch.sigmoid(outputs) > 0.5).float()
            
            for i in range(images.size(0)):
                iou = iou_metric(preds[i].unsqueeze(0).int(), masks[i].unsqueeze(0).int()).item()
                
                if tumors_type[i] == 'gl':
                    gl_iou.append(iou)
                elif tumors_type[i] == 'me':
                    me_iou.append(iou)
                elif tumors_type[i] == 'pi':
                    pi_iou.append(iou)
    
    len_gl = len(gl_iou)
    len_me = len(me_iou)
    len_pi = len(pi_iou)
    total_samples = len_gl + len_me + len_pi
    
    gl_miou = np.mean(gl_iou) if len_gl > 0 else 0
    me_miou = np.mean(me_iou) if len_me > 0 else 0
    pi_miou = np.mean(pi_iou) if len_pi > 0 else 0
    weighted_miou = (gl_miou * len_gl + me_miou * len_me + pi_miou * len_pi) / total_samples if total_samples > 0 else 0
    
    print(f"\n{model_name} Results:")
    print(f"GL: {gl_miou:.4f} ({len_gl}) | ME: {me_miou:.4f} ({len_me}) | PI: {pi_miou:.4f} ({len_pi})")
    print(f"Weighted mIoU: {weighted_miou:.4f}")
    
    return {
        'gl': gl_miou,
        'me': me_miou,
        'pi': pi_miou,
        'weighted': weighted_miou
    }


In [18]:
# ============================================
# 13. ENSEMBLE EVALUATION
# ============================================
print("\n" + "="*70)
print("ENSEMBLE EVALUATION - COMPARING ALL MODELS")
print("="*70)

# Test different fusion strategies
fusion_methods = ['average', 'weighted', 'max']
val_results = {}
test_results = {}

for method in fusion_methods:
    print(f"\n{'='*70}")
    print(f"Testing Fusion Method: {method.upper()}")
    print(f"{'='*70}")
    
    ensemble = EnsembleModel(
        unet_path='best_unet_model.pth',
        cnn_path='best_cnn_model.pth',
        device=device,
        fusion_method=method
    )
    
    # Validation set
    print("\n--- VALIDATION SET ---")
    val_results[method] = evaluate_model(ensemble, val_loader, device, f"Ensemble ({method})")
    
    # Test set
    print("\n--- TEST SET ---")
    test_results[method] = evaluate_model(ensemble, test_loader, device, f"Ensemble ({method})")
    
    # Clean up
    del ensemble
    torch.cuda.empty_cache()


ENSEMBLE EVALUATION - COMPARING ALL MODELS

Testing Fusion Method: AVERAGE
Loading U-Net...
✓ U-Net loaded (Val mIoU: 0.7699)
Loading CNN...
✓ CNN loaded (Val mIoU: 0.6779)

--- VALIDATION SET ---


Evaluating Ensemble (average): 100%|██████████| 50/50 [00:06<00:00,  8.12it/s]



Ensemble (average) Results:
GL: 0.6259 (229) | ME: 0.8683 (266) | PI: 0.7759 (292)
Weighted mIoU: 0.7635

--- TEST SET ---


Evaluating Ensemble (average): 100%|██████████| 54/54 [00:06<00:00,  7.89it/s]



Ensemble (average) Results:
GL: 0.6259 (254) | ME: 0.8816 (306) | PI: 0.7689 (300)
Weighted mIoU: 0.7668

Testing Fusion Method: WEIGHTED
Loading U-Net...
✓ U-Net loaded (Val mIoU: 0.7699)
Loading CNN...
✓ CNN loaded (Val mIoU: 0.6779)

--- VALIDATION SET ---


Evaluating Ensemble (weighted): 100%|██████████| 50/50 [00:06<00:00,  8.13it/s]


... (3 outputs left out)

Evaluating Ensemble (max): 100%|██████████| 50/50 [00:06<00:00,  7.98it/s]



Ensemble (max) Results:
GL: 0.6221 (229) | ME: 0.8258 (266) | PI: 0.7410 (292)
Weighted mIoU: 0.7350

--- TEST SET ---


Evaluating Ensemble (max): 100%|██████████| 54/54 [00:06<00:00,  7.93it/s]



Ensemble (max) Results:
GL: 0.6173 (254) | ME: 0.8560 (306) | PI: 0.7191 (300)
Weighted mIoU: 0.7377


In [24]:
# ============================================
# 14. FINAL COMPARISON
# ============================================
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

print("\n--- VALIDATION SET ---")
print(f"{'Method':<15} {'GL mIoU':<10} {'ME mIoU':<10} {'PI mIoU':<10} {'Weighted mIoU':<15}")
print("-" * 70)
for method, res in val_results.items():
    print(f"{method.capitalize():<15} {res['gl']:<10.4f} {res['me']:<10.4f} {res['pi']:<10.4f} {res['weighted']:<15.4f}")

print("\n--- TEST SET ---")
print(f"{'Method':<15} {'GL mIoU':<10} {'ME mIoU':<10} {'PI mIoU':<10} {'Weighted mIoU':<15}")
print("-" * 70)
for method, res in test_results.items():
    print(f"{method.capitalize():<15} {res['gl']:<10.4f} {res['me']:<10.4f} {res['pi']:<10.4f} {res['weighted']:<15.4f}")

# Find best method
best_val_method = max(val_results.items(), key=lambda x: x[1]['weighted'])
best_test_method = max(test_results.items(), key=lambda x: x[1]['weighted'])

print("\n" + "="*70)
print(f"Best Validation Method: {best_val_method[0].upper()} (mIoU: {best_val_method[1]['weighted']:.4f})")
print(f"Best Test Method: {best_test_method[0].upper()} (mIoU: {best_test_method[1]['weighted']:.4f})")
print("="*70)


FINAL RESULTS SUMMARY

--- VALIDATION SET ---
Method          GL mIoU    ME mIoU    PI mIoU    Weighted mIoU  
----------------------------------------------------------------------
Average         0.6259     0.8683     0.7759     0.7635         
Weighted        0.6324     0.8713     0.7825     0.7688         
Max             0.6221     0.8258     0.7410     0.7350         

--- TEST SET ---
Method          GL mIoU    ME mIoU    PI mIoU    Weighted mIoU  
----------------------------------------------------------------------
Average         0.6259     0.8816     0.7689     0.7668         
Weighted        0.6338     0.8846     0.7729     0.7715         
Max             0.6173     0.8560     0.7191     0.7377         

Best Validation Method: WEIGHTED (mIoU: 0.7688)
Best Test Method: WEIGHTED (mIoU: 0.7715)
